# Business Entity Resolution Challenge — Interactive Pipeline

**Objective:** Resolve noisy business entity records across three independent sources (`Source 1`, `Source 2`, `Source 3`) to maximize **Macro F0.5 Score** per Source 1 entity.

$$\text{F}_{0.5} = \frac{1.25 \times \text{Precision} \times \text{Recall}}{0.25 \times \text{Precision} + \text{Recall}}$$

* Singletons score **1.0** when correctly predicted as empty, and **0.0** on any false merge.
* This notebook executes every stage of the pipeline sequentially: **Data Loading $\to$ Preprocessing $\to$ Blocking $\to$ Feature Engineering $\to$ Model Training (GroupKFold CV) $\to$ Threshold Tuning $\to$ Inference $\to$ Validation $\to$ Packaging**.

## 1. Environment Setup & Configuration

In [ ]:
import os
import sys
import time
import json
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm

# Ensure project modules are on path
ROOT_DIR = os.path.abspath(".")
SRC_DIR = os.path.join(ROOT_DIR, "code", "business_entity_resolution")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

TRAIN_DIR = os.path.join(ROOT_DIR, "student_resource", "dataset", "train")
TEST_DIR = os.path.join(ROOT_DIR, "student_resource", "dataset", "test")
OUTPUT_DIR = os.path.join(ROOT_DIR, "output")
MODEL_DIR = os.path.join(ROOT_DIR, "models")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Set SAMPLE_SIZE: Set to an integer (e.g. 5000) for fast prototyping, or None for full scale
SAMPLE_SIZE = 5000
TOP_K_CANDIDATES = 80
CV_SPLITS = 5

print(f"Setup complete!")
print(f"Train directory: {TRAIN_DIR}")
print(f"Test directory:  {TEST_DIR}")
print(f"Sample size:     {SAMPLE_SIZE} (Set SAMPLE_SIZE = None for full 2.2M dataset)")

## 2. Load Datasets & Ground Truth

In [ ]:
from src.utils import load_tsv, load_ground_truth, sample_representative_dataset

if SAMPLE_SIZE and SAMPLE_SIZE > 0:
    print(f"Sampling representative training dataset ({SAMPLE_SIZE:,} entities) preserving true 5.58% singleton rate...")
    s1_train, s2_train, s3_train, gt_train = sample_representative_dataset(
        train_dir=TRAIN_DIR,
        sample_size=SAMPLE_SIZE,
        random_state=42,
    )
else:
    print("Loading full training datasets...")
    s1_train = load_tsv(os.path.join(TRAIN_DIR, "train_source1.tsv"))
    s2_train = load_tsv(os.path.join(TRAIN_DIR, "train_source2.tsv"))
    s3_train = load_tsv(os.path.join(TRAIN_DIR, "train_source3.tsv"))
    gt_train = load_ground_truth(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"))

print(f"Loaded Train S1: {len(s1_train):,} entities")
print(f"Loaded Train S2: {len(s2_train):,} entities")
print(f"Loaded Train S3: {len(s3_train):,} entities")
print(f"Loaded Ground Truth S1 keys: {len(gt_train):,}")


## 3. High-Speed Preprocessing & Normalization
* Unicode NFKC normalization
* Diacritic/accent removal (crucial for French entities in test data)
* Expansion of legal suffixes (`pvt` $\to$ `private`, `ltd` $\to$ `limited`, etc.)
* Street abbreviation standardization (`rd` $\to$ `road`, `st` $\to$ `street`)
* Extraction of core names and numeric tokens (postal codes / building numbers)

In [ ]:
from src.preprocessing import preprocess_dataframe

t0 = time.time()
print("Preprocessing Train S1...")
s1_train = preprocess_dataframe(s1_train)
print("Preprocessing Train S2...")
s2_train = preprocess_dataframe(s2_train)
print("Preprocessing Train S3...")
s3_train = preprocess_dataframe(s3_train)
print(f"Preprocessing complete in {time.time() - t0:.2f}s!")

# Display sample normalized records
display_cols = ["entity_id", "norm_name", "core_name", "norm_address", "address_numbers", "norm_country"]
s1_train[display_cols].head(3)

## 4. Multi-View Candidate Generation (Blocking)
* Character $n$-gram TF-IDF inverted index
* Token & numeric address inverted index
* Open-set country gating (relational matching without hardcoding countries)

In [ ]:
from src.blocking import CandidateBlocker
from src.evaluation import compute_candidate_recall

t0 = time.time()
blocker = CandidateBlocker(top_k=TOP_K_CANDIDATES)
train_candidates = blocker.block(s1_train, s2_train, s3_train)
train_cand_map = blocker.candidate_df_to_mapping(train_candidates, list(s1_train["entity_id"]))

recall_stats = compute_candidate_recall(gt_train, train_cand_map)
print(f"Blocking complete in {time.time() - t0:.2f}s!")
print(f"Total Candidates Generated: {len(train_candidates):,}")
print(f"Candidate Recall: {recall_stats['candidate_recall'] * 100:.2f}% ({recall_stats['retrieved_true_pairs']}/{recall_stats['total_true_pairs']} true pairs captured)")
print(f"Average Candidates per S1: {recall_stats['avg_candidates_per_s1']:.1f}")

## 5. Pairwise Feature Engineering
Extracts 35 discriminative features for every candidate pair $(S1, C)$:
* Fuzzy edit similarities (Levenshtein, token sort, token set, partial ratio)
* Character 3-gram & word token Jaccard similarities
* Address numeric invariants (postal code and house number exact match)
* Open-set relational country indicators (`country_exact_match`, `country_mismatch`, `country_missing_either`)

In [ ]:
from src.train import prepare_training_pairs
from src.features import FeatureExtractor

t0 = time.time()
train_pairs_df, labels = prepare_training_pairs(
    s1_df=s1_train,
    candidate_pairs_df=train_candidates,
    ground_truth_map=gt_train,
    negative_ratio=10.0,
)

print("Extracting pairwise feature matrix...")
extractor = FeatureExtractor()
X_train = extractor.extract_features(train_pairs_df, s1_train, s2_train, s3_train)
feature_names = list(X_train.columns)

print(f"Feature matrix extracted in {time.time() - t0:.2f}s: {X_train.shape[0]:,} rows x {X_train.shape[1]} features!")

## 6. GroupKFold Cross-Validation & Macro F0.5 Optimization
* Cross-validation partitioned strictly by `source1_entity_id` to prevent data leakage.
* Threshold swept across $\theta \in [0.20, 0.84]$ with 1-to-1 conflict resolution to directly maximize the official Macro F0.5 metric.

In [ ]:
from src.train import split_group_kfold, optimize_threshold
from src.models import EntityMatchingModel

groups = train_pairs_df["source1_entity_id"].values
oof_preds = np.zeros(len(train_pairs_df))

print(f"Running {CV_SPLITS}-Fold GroupKFold Cross-Validation...")
for fold, (t_idx, v_idx) in enumerate(split_group_kfold(groups, n_splits=CV_SPLITS), start=1):
    fold_model = EntityMatchingModel(model_type="lightgbm")
    fold_model.fit(X_train.iloc[t_idx], labels[t_idx], feature_names=feature_names)
    oof_preds[v_idx] = fold_model.predict_proba(X_train.iloc[v_idx])
    print(f"  -> Fold {fold}/{CV_SPLITS} complete.")

oof_df = train_pairs_df[["source1_entity_id", "candidate_entity_id"]].copy()
oof_df["probability"] = oof_preds

all_s1_ids = list(s1_train["entity_id"])
best_threshold, best_f05, best_metrics = optimize_threshold(
    oof_df=oof_df,
    ground_truth_map=gt_train,
    all_s1_ids=all_s1_ids,
    enforce_one_to_one=True,
)

print("\n" + "="*50)
print(f"OPTIMAL THRESHOLD: {best_threshold:.3f}")
print(f"MACRO F0.5 SCORE:  {best_f05:.4f}")
print(f"Macro Precision:   {best_metrics.get('macro_precision', 0):.4f}")
print(f"Macro Recall:      {best_metrics.get('macro_recall', 0):.4f}")
print(f"Singleton Acc:     {best_metrics.get('singleton_accuracy', 0):.4f}")
print("="*50)

## 7. Fit Production Model & View Feature Importances

In [ ]:
final_model = EntityMatchingModel(model_type="lightgbm")
final_model.fit(X_train, labels, feature_names=feature_names)

# Save model and metadata
model_path = os.path.join(MODEL_DIR, "entity_matching_model.joblib")
final_model.save(model_path)
print(f"Model serialized to {model_path}")

# Display Top 10 Features
importances = final_model.get_feature_importances()
imp_df = pd.DataFrame(list(importances.items())[:10], columns=["Feature", "Importance"])
imp_df

## 8. Test Inference & Output Generation
Generates:
1. `output/candidate_pairs.tsv`
2. `output/matching_results.tsv` (Leaderboard submission)

In [ ]:
from src.predict import predict_pipeline

metadata_path = os.path.join(MODEL_DIR, "model_metadata.json")
print("Running memory-safe prediction pipeline on test datasets (chunked inference)...")
out_paths = predict_pipeline(
    test_dir=TEST_DIR,
    model_path=model_path,
    metadata_path=metadata_path,
    output_dir=OUTPUT_DIR,
    top_k_candidates=TOP_K_CANDIDATES,
    threshold=best_threshold,
    enforce_one_to_one=True,
    chunk_size=50000,
)

cand_file = out_paths["candidate_file"]
matching_file = out_paths["matching_file"]
print(f"\nTest inference complete!")
print(f"Candidate pairs written to: {cand_file}")
print(f"Matching results written to: {matching_file}")


## 9. Validate Submission TSVs
Runs the official `validate_submission.py` to confirm zero format or integrity errors.

In [ ]:
validator_script = os.path.join(ROOT_DIR, "utils", "validate_submission.py")
if not os.path.exists(validator_script):
    validator_script = os.path.join(ROOT_DIR, "student_resource", "utils", "validate_submission.py")

print("Running official submission validator...")
!python "{validator_script}" --matching "{matching_file}" --candidate "{cand_file}" --test-dir "{TEST_DIR}"

## 10. Package Final Submission Zip

In [ ]:
packager_script = os.path.join(ROOT_DIR, "package_submission.py")
!python "{packager_script}" --team-name "Code_Alchemists" --test-dir "{TEST_DIR}"